## The following is a download of PositioNZ data and subsequent comparison with data downloaded from IGS Helmert files.

In [ ]:
import seaborn as sns
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
import os 
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt
import re
from typing import Dict
from pathlib import Path

#### Set up the date ranges

In [ ]:
# Define time range
start = datetime(2025, 4, 10)
end = datetime(2025, 10, 27)

In [ ]:
# Define station groups
PNcodes=['AUCK', 'DUND','MQZG','WGTN']

In [ ]:
# Calculate crd_epoch
crd_epoch = end + (start - end) / 2
astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear

In [ ]:
# --- Clear, human-readable labels for solution types ---
SOLUTION_LABEL_MAP = {
    'd20f_54_code_A':   'GPS+Galileo+GLONASS',  
}

# Initialise a dictionary to store DataFrames for each solution type
daily_avg_dfs = {
    # Keep keys as your original solution types for API calls and grouping
        
    'd20f_54_code_A':   {},  # GPS + Galileo + GLONASS  
}

# Loop through PNcodes and solution types
for code in PNcodes:
    for sol_type in daily_avg_dfs.keys():
        try:
            print(f"Processing station: {code} with solution type: {sol_type}")
            ts = CoordApiTimeseries(code, solutiontype=sol_type, after=start, before=end)
            dates, xyz = ts.withoutOutliers().getObs(enu=False)

            df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
            df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
            df_obs['station'] = code

            # Apply the human-readable label
            df_obs['solutiontype'] = SOLUTION_LABEL_MAP.get(sol_type, sol_type)

            daily_avg_dfs[sol_type][code] = df_obs

        except Exception as e:
            print(f"Failed to process station {code} with solution type {sol_type}. Error: {e}")

# Combine all DataFrames into one
PN_data = pd.concat(
    [pd.concat(dfs.values(), axis=0) for dfs in daily_avg_dfs.values()],
    axis=0
)

# Sort alphabetically by station, date, and solution type
PN_data.sort_values(by=["station", "date", "solutiontype"], inplace=True)

# Print confirmation
print("Download complete ✅")
print("PN_data DataFrame created with shape:", PN_data.shape)
print("Columns:", PN_data.columns.tolist())
print("Solution types present:", sorted(PN_data['solutiontype'].unique()))

In [ ]:
# Group PN_data by station and solutiontype
group_dataframes = {
    f"{station}_{sol_type}": group
    for (station, sol_type), group in PN_data.groupby(["station", "solutiontype"])
}

In [ ]:
# Define expected converted DataFrame names
converted_columns = [
    'x', 'y', 'z', 'date', 'station',
    'nztm_easting', 'nztm_northing', 'nzvd2016_elev'
]

# Initialize empty dictionary with expected keys
empty_converted_dfs = {
    f"{name}_converted": pd.DataFrame(columns=converted_columns)
    for name in group_dataframes.keys()
}

In [ ]:
converted_group_dataframes = {}

def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"}

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"❌ API conversion failed with status {response.status_code}")
        print(f"🔍 Error details: {response.text}")
        return []

In [ ]:
for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    # Always start with a fresh copy
    df = original_df.copy(deep=True)

    # Identify and print NaN values
    nan_values = df[df.isna().any(axis=1)]
    if not nan_values.empty:
        print(f" - Found {len(nan_values)} NaN rows")

    # Identify and print Infinity values
    inf_values = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_values.empty:
        print(f" - Found {len(inf_values)} Inf rows")

    # Filter out invalid rows
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Prepare coordinates
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()

    # Convert coordinates
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    # Add converted coordinates to DataFrame
    if converted_coords:
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )

        # Store in the pre-initialised DataFrame
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")

In [ ]:
EXPORT_DIR = Path("Exports")
EXPORT_DIR.mkdir(exist_ok=True)

all_converted_rows = []

for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    df = original_df.copy(deep=True)

    # Identify NaN / Inf (optional debug prints)
    nan_rows = df[df.isna().any(axis=1)]
    if not nan_rows.empty:
        print(f" - Found {len(nan_rows)} NaN rows")
    inf_rows = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_rows.empty:
        print(f" - Found {len(inf_rows)} Inf rows")

    # Filter invalid coordinates
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Convert coordinates (expects list of [x,y,z])
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    if converted_coords:
        # Add converted columns
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )
        df_filtered['group_name'] = name  # traceability

        # Save per-group CSV
        per_group_csv = EXPORT_DIR / f"{name}_converted_coords.csv"
        df_filtered.to_csv(per_group_csv, index=False)
        print(f"📄 Saved per-group CSV: {per_group_csv.resolve()}")

        # Update the pre-initialised store (if present)
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")

        # Accumulate for combined CSV
        all_converted_rows.append(df_filtered)
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")

# --- Combined CSV across all groups ---
if all_converted_rows:
    combined_df = pd.concat(all_converted_rows, ignore_index=True)

    # Order columns neatly if they exist
    preferred_cols = [
        "group_name", "x", "y", "z", "nztm_easting", "nztm_northing", "nzvd2016_elev"
    ]
    cols = [c for c in preferred_cols if c in combined_df.columns] + \
           [c for c in combined_df.columns if c not in preferred_cols]
    combined_df = combined_df[cols]

    combined_csv = EXPORT_DIR / "nz_all_converted_coordinates.csv"
    combined_df.to_csv(combined_csv, index=False)
    print(f"\n📦 Saved combined CSV for all groups: {combined_csv.resolve()}")
else:
    print


In [ ]:
def calculate_3d_rms(df, coord_cols=['nztm_easting', 'nztm_northing', 'nzvd2016_elev']):
    """
    Calculate 3-D RMS for a DataFrame containing coordinate observations.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with coordinate columns
    coord_cols : list
        Column names for E, N, U components
    
    Returns:
    --------
    float : 3-D RMS value in meters
    """
    # Calculate mean position
    mean_e = df[coord_cols[0]].mean()
    mean_n = df[coord_cols[1]].mean()
    mean_u = df[coord_cols[2]].mean()
    
    # Calculate residuals from mean
    residuals_e = df[coord_cols[0]] - mean_e
    residuals_n = df[coord_cols[1]] - mean_n
    residuals_u = df[coord_cols[2]] - mean_u
    
    # Calculate standard deviation for each component
    sigma_e = residuals_e.std()
    sigma_n = residuals_n.std()
    sigma_u = residuals_u.std()
    
    # Calculate 3-D RMS
    rms_3d = np.sqrt(sigma_e**2 + sigma_n**2 + sigma_u**2)
    
    return rms_3d


def calculate_daily_3d_rms(row, coord_cols=['nztm_easting', 'nztm_northing', 'nzvd2016_elev']):
    """
    Calculate 3-D RMS for a single day's observation.
    Since we only have one observation per day, RMS is calculated from the 
    residual components directly.
    
    Parameters:
    -----------
    row : pd.Series
        Single row with coordinate columns
    coord_cols : list
        Column names for E, N, U components
    
    Returns:
    --------
    float : 3-D magnitude in meters
    """
    # For a single observation, we can't calculate std dev
    # So we'll need to calculate this differently - see note below
    return np.nan


# ========================================================================
# OVERALL RMS CALCULATION (Original approach - Option C)
# ========================================================================

print("="*70)
print("OVERALL 3-D RMS CALCULATION")
print("="*70)

# Calculate PositioNZ-PP RMS for each station (across all days)
pn_rms_results = []

for name, df in empty_converted_dfs.items():
    if df.empty or 'nztm_easting' not in df.columns:
        print(f"Skipping {name} - no converted coordinates")
        continue
    
    station = df['station'].iloc[0] if 'station' in df.columns else name.split('_')[0]
    rms_3d = calculate_3d_rms(df)
    
    pn_rms_results.append({
        'station': station,
        'rms_3d_mm': rms_3d * 1000,
        'n_obs': len(df)
    })
    
    print(f"{station}: 3-D RMS = {rms_3d*1000:.2f} mm (n={len(df)} observations)")

pn_rms = pd.DataFrame(pn_rms_results)
pn_rms['method'] = 'PositioNZ-PP'


# ========================================================================
# DAILY RMS CALCULATION (Option A - Day-by-day comparison)
# ========================================================================

print("\n" + "="*70)
print("DAILY 3-D RMS CALCULATION")
print("="*70)

# For PositioNZ-PP: Calculate daily RMS
# Note: With one observation per day, we need to define what "daily RMS" means
# Option 1: Daily deviation from overall mean position
# Option 2: Use the r_mag_mm approach (magnitude of residual from daily solution)

pn_daily_rms_results = []

for name, df in empty_converted_dfs.items():
    if df.empty or 'nztm_easting' not in df.columns:
        continue
    
    station = df['station'].iloc[0] if 'station' in df.columns else name.split('_')[0]
    
    # Calculate overall mean position
    mean_e = df['nztm_easting'].mean()
    mean_n = df['nztm_northing'].mean()
    mean_u = df['nzvd2016_elev'].mean()
    
    # For each day, calculate deviation from mean
    for idx, row in df.iterrows():
        de = (row['nztm_easting'] - mean_e) * 1000  # Convert to mm
        dn = (row['nztm_northing'] - mean_n) * 1000
        du = (row['nzvd2016_elev'] - mean_u) * 1000
        
        # Calculate 3-D magnitude
        rms_3d_mm = np.sqrt(de**2 + dn**2 + du**2)
        
        pn_daily_rms_results.append({
            'date': row['date'],
            'station': station,
            'rms_mm': rms_3d_mm,
            'method': 'PositioNZ-PP'
        })

pn_daily_rms = pd.DataFrame(pn_daily_rms_results)

print(f"\nPositioNZ-PP daily RMS calculated for {len(pn_daily_rms)} station-days")
print("\nFirst 10 rows:")
print(pn_daily_rms.head(10).to_string(index=False))


# ========================================================================
# LOAD IGS DAILY RMS DATA
# ========================================================================

print("\n" + "="*70)
print("LOADING IGS HELMERT DAILY RMS DATA")
print("="*70)

try:
    # Load the pre-calculated daily RMS file
    igs_daily_data = pd.read_csv('exports/nz_daily_site_rms.csv')
    
    # Rename columns to match our format
    igs_daily_rms = igs_daily_data[['date', 'site', 'rms_mm']].copy()
    igs_daily_rms.rename(columns={'site': 'station'}, inplace=True)
    igs_daily_rms['method'] = 'IGS Helmert'
    
    print(f"\nIGS Helmert daily RMS loaded for {len(igs_daily_rms)} station-days")
    print("\nFirst 10 rows:")
    print(igs_daily_rms.head(10).to_string(index=False))
    
except Exception as e:
    print(f"Error loading IGS data: {e}")
    igs_daily_rms = None


# ========================================================================
# OPTION A: DAY-BY-DAY COMPARISON
# ========================================================================

if igs_daily_rms is not None:
    print("\n" + "="*70)
    print("OPTION A: DAY-BY-DAY COMPARISON")
    print("="*70)
    
    # Merge daily RMS values
    daily_comparison = pd.merge(
        pn_daily_rms[['date', 'station', 'rms_mm']].rename(columns={'rms_mm': 'PositioNZ_RMS_mm'}),
        igs_daily_rms[['date', 'station', 'rms_mm']].rename(columns={'rms_mm': 'IGS_RMS_mm'}),
        on=['date', 'station'],
        how='inner'
    )
    
    daily_comparison['difference_mm'] = daily_comparison['PositioNZ_RMS_mm'] - daily_comparison['IGS_RMS_mm']
    daily_comparison['ratio'] = daily_comparison['PositioNZ_RMS_mm'] / daily_comparison['IGS_RMS_mm']
    
    print(f"\nMatched {len(daily_comparison)} station-days")
    print("\nFirst 20 rows:")
    print(daily_comparison.head(20).to_string(index=False))
    
    # Save daily comparison
    daily_comparison.to_csv('exports/daily_rms_comparison.csv', index=False)
    print("\nâœ… Daily comparison saved to exports/daily_rms_comparison.csv")


# ========================================================================
# OPTION B: SUMMARY STATISTICS COMPARISON
# ========================================================================

if igs_daily_rms is not None:
    print("\n" + "="*70)
    print("OPTION B: SUMMARY STATISTICS OF DAILY RMS")
    print("="*70)
    
    # Calculate summary stats for PositioNZ-PP
    pn_stats = pn_daily_rms.groupby('station')['rms_mm'].agg([
        ('mean_daily_rms', 'mean'),
        ('median_daily_rms', 'median'),
        ('std_daily_rms', 'std'),
        ('n_days', 'count')
    ]).reset_index()
    pn_stats['method'] = 'PositioNZ-PP'
    
    # Calculate summary stats for IGS
    igs_stats = igs_daily_rms.groupby('station')['rms_mm'].agg([
        ('mean_daily_rms', 'mean'),
        ('median_daily_rms', 'median'),
        ('std_daily_rms', 'std'),
        ('n_days', 'count')
    ]).reset_index()
    igs_stats['method'] = 'IGS Helmert'
    
    print("\nPositioNZ-PP Summary Statistics:")
    print(pn_stats.to_string(index=False))
    
    print("\n\nIGS Helmert Summary Statistics:")
    print(igs_stats.to_string(index=False))
    
    # Create comparison
    stats_comparison = pd.merge(
        pn_stats[['station', 'mean_daily_rms', 'median_daily_rms', 'std_daily_rms']],
        igs_stats[['station', 'mean_daily_rms', 'median_daily_rms', 'std_daily_rms']],
        on='station',
        suffixes=('_PositioNZ', '_IGS')
    )
    
    stats_comparison['mean_ratio'] = stats_comparison['mean_daily_rms_PositioNZ'] / stats_comparison['mean_daily_rms_IGS']
    stats_comparison['median_ratio'] = stats_comparison['median_daily_rms_PositioNZ'] / stats_comparison['median_daily_rms_IGS']
    
    print("\n\nSummary Statistics Comparison:")
    print(stats_comparison.to_string(index=False))
    
    # Save summary comparison
    stats_comparison.to_csv('exports/summary_stats_comparison.csv', index=False)
    print("\nâœ… Summary statistics saved to exports/summary_stats_comparison.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Combine both datasets for plotting
combined_daily = pd.concat([pn_daily_rms, igs_daily_rms], ignore_index=True)

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Box plot by station
sns.boxplot(data=combined_daily, x='station', y='rms_mm', hue='method', ax=axes[0])
axes[0].set_title('Daily 3-D RMS by Station', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Station', fontsize=12)
axes[0].set_ylabel('3-D RMS (mm)', fontsize=12)
axes[0].legend(title='Method')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Box plot by method (all stations combined)
sns.boxplot(data=combined_daily, x='method', y='rms_mm', ax=axes[1])
axes[1].set_title('Daily 3-D RMS by Processing Method', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Processing Method', fontsize=12)
axes[1].set_ylabel('3-D RMS (mm)', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('exports/daily_rms_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Box plot saved to exports/daily_rms_boxplot.png")

# Optional: Create individual plots per station for more detail
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

stations = combined_daily['station'].unique()
for i, station in enumerate(stations):
    if i < 4:  # Only plot if we have up to 4 stations
        station_data = combined_daily[combined_daily['station'] == station]
        sns.boxplot(data=station_data, x='method', y='rms_mm', ax=axes[i])
        axes[i].set_title(f'{station}', fontsize=12, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('3-D RMS (mm)', fontsize=10)
        axes[i].grid(axis='y', alpha=0.3)

# Hide unused subplots
for i in range(len(stations), 4):
    axes[i].set_visible(False)

plt.suptitle('Daily 3-D RMS Comparison by Station', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('exports/daily_rms_boxplot_per_station.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Per-station box plots saved to exports/daily_rms_boxplot_per_station.png")

# Print summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
summary = combined_daily.groupby(['station', 'method'])['rms_mm'].describe()
print(summary)

In [ ]:
# Load the AUCK data
auck = pd.read_csv('Exports/AUCK_GPS+Galileo+GLONASS_converted_coords.csv')

print("="*70)
print("AUCK DATA OVERVIEW")
print("="*70)
print(f"\nTotal observations: {len(auck)}")
print(f"Date range: {auck['date'].min()} to {auck['date'].max()}")
print(f"\nColumns: {auck.columns.tolist()}")

# Convert date to datetime for time series analysis
auck['datetime'] = pd.to_datetime(auck['date'])

print("\n" + "="*70)
print("COORDINATE STATISTICS")
print("="*70)

# ITRF2020 XYZ statistics
print("\nITRF2020 XYZ Coordinates (meters):")
print(f"X: mean = {auck['x'].mean():.4f}, std = {auck['x'].std():.6f}")
print(f"Y: mean = {auck['y'].mean():.4f}, std = {auck['y'].std():.6f}")
print(f"Z: mean = {auck['z'].mean():.4f}, std = {auck['z'].std():.6f}")

# NZTM coordinates statistics
print("\nNZTM Coordinates (meters):")
print(f"Easting:  mean = {auck['nztm_easting'].mean():.4f}, std = {auck['nztm_easting'].std():.6f}")
print(f"Northing: mean = {auck['nztm_northing'].mean():.4f}, std = {auck['nztm_northing'].std():.6f}")
print(f"Elevation: mean = {auck['nzvd2016_elev'].mean():.4f}, std = {auck['nzvd2016_elev'].std():.6f}")

# Calculate residuals from mean position
print("\n" + "="*70)
print("RESIDUALS FROM MEAN POSITION")
print("="*70)

# NZTM residuals
mean_e = auck['nztm_easting'].mean()
mean_n = auck['nztm_northing'].mean()
mean_u = auck['nzvd2016_elev'].mean()

auck['de_mm'] = (auck['nztm_easting'] - mean_e) * 1000
auck['dn_mm'] = (auck['nztm_northing'] - mean_n) * 1000
auck['du_mm'] = (auck['nzvd2016_elev'] - mean_u) * 1000

print("\nResidual Statistics (mm):")
print(f"East:  std = {auck['de_mm'].std():.3f} mm")
print(f"North: std = {auck['dn_mm'].std():.3f} mm")
print(f"Up:    std = {auck['du_mm'].std():.3f} mm")

# Calculate 3-D RMS
sigma_e = auck['de_mm'].std()
sigma_n = auck['dn_mm'].std()
sigma_u = auck['du_mm'].std()
rms_3d = np.sqrt(sigma_e**2 + sigma_n**2 + sigma_u**2)

print(f"\n3-D RMS: {rms_3d:.3f} mm")

# Component contributions
print(f"\nComponent contributions to 3-D RMS:")
print(f"East:  {(sigma_e**2 / rms_3d**2) * 100:.1f}%")
print(f"North: {(sigma_n**2 / rms_3d**2) * 100:.1f}%")
print(f"Up:    {(sigma_u**2 / rms_3d**2) * 100:.1f}%")

# Calculate daily 3-D magnitude
auck['rms_daily_mm'] = np.sqrt(auck['de_mm']**2 + auck['dn_mm']**2 + auck['du_mm']**2)

print("\n" + "="*70)
print("DAILY 3-D RMS STATISTICS")
print("="*70)
print(f"Mean:   {auck['rms_daily_mm'].mean():.3f} mm")
print(f"Median: {auck['rms_daily_mm'].median():.3f} mm")
print(f"Std:    {auck['rms_daily_mm'].std():.3f} mm")
print(f"Min:    {auck['rms_daily_mm'].min():.3f} mm")
print(f"Max:    {auck['rms_daily_mm'].max():.3f} mm")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Time series of residuals
ax = axes[0, 0]
ax.plot(auck['datetime'], auck['de_mm'], label='East', alpha=0.7)
ax.plot(auck['datetime'], auck['dn_mm'], label='North', alpha=0.7)
ax.plot(auck['datetime'], auck['du_mm'], label='Up', alpha=0.7)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Residuals (mm)')
ax.set_title('AUCK: Coordinate Residuals Over Time')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Daily 3-D RMS over time
ax = axes[0, 1]
ax.plot(auck['datetime'], auck['rms_daily_mm'], color='purple', alpha=0.7)
ax.axhline(auck['rms_daily_mm'].mean(), color='red', linestyle='--', 
           label=f'Mean: {auck["rms_daily_mm"].mean():.2f} mm')
ax.set_xlabel('Date')
ax.set_ylabel('Daily 3-D RMS (mm)')
ax.set_title('AUCK: Daily 3-D Positional Scatter')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: Horizontal scatter plot
ax = axes[1, 0]
ax.scatter(auck['de_mm'], auck['dn_mm'], alpha=0.5, s=20)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.axvline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('East Residual (mm)')
ax.set_ylabel('North Residual (mm)')
ax.set_title('AUCK: Horizontal Position Scatter')
ax.axis('equal')
ax.grid(alpha=0.3)

# Plot 4: Histogram of daily RMS
ax = axes[1, 1]
ax.hist(auck['rms_daily_mm'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(auck['rms_daily_mm'].mean(), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {auck["rms_daily_mm"].mean():.2f} mm')
ax.axvline(auck['rms_daily_mm'].median(), color='blue', linestyle='--', 
           linewidth=2, label=f'Median: {auck["rms_daily_mm"].median():.2f} mm')
ax.set_xlabel('Daily 3-D RMS (mm)')
ax.set_ylabel('Frequency')
ax.set_title('AUCK: Distribution of Daily 3-D RMS')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exports/auck_coordinate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Analysis complete! Plot saved to exports/auck_coordinate_analysis.png")

# Check for outliers (>3 sigma)
outliers = auck[auck['rms_daily_mm'] > auck['rms_daily_mm'].mean() + 3*auck['rms_daily_mm'].std()]
if len(outliers) > 0:
    print(f"\n⚠️  Found {len(outliers)} potential outlier days (>3σ):")
    print(outliers[['date', 'rms_daily_mm']].to_string(index=False))

In [ ]:
# Load the MQZG data
mqzg = pd.read_csv('Exports/MQZG_GPS+Galileo+GLONASS_converted_coords.csv')

print("="*70)
print("MQZG DATA OVERVIEW")
print("="*70)
print(f"\nTotal observations: {len(mqzg)}")
print(f"Date range: {mqzg['date'].min()} to {mqzg['date'].max()}")
print(f"\nColumns: {mqzg.columns.tolist()}")

# Convert date to datetime for time series analysis
mqzg['datetime'] = pd.to_datetime(mqzg['date'])

print("\n" + "="*70)
print("COORDINATE STATISTICS")
print("="*70)

# ITRF2020 XYZ statistics
print("\nITRF2020 XYZ Coordinates (meters):")
print(f"X: mean = {mqzg['x'].mean():.4f}, std = {mqzg['x'].std():.6f}")
print(f"Y: mean = {mqzg['y'].mean():.4f}, std = {mqzg['y'].std():.6f}")
print(f"Z: mean = {mqzg['z'].mean():.4f}, std = {mqzg['z'].std():.6f}")

# NZTM coordinates statistics
print("\nNZTM Coordinates (meters):")
print(f"Easting:  mean = {mqzg['nztm_easting'].mean():.4f}, std = {mqzg['nztm_easting'].std():.6f}")
print(f"Northing: mean = {mqzg['nztm_northing'].mean():.4f}, std = {mqzg['nztm_northing'].std():.6f}")
print(f"Elevation: mean = {mqzg['nzvd2016_elev'].mean():.4f}, std = {mqzg['nzvd2016_elev'].std():.6f}")

# Calculate residuals from mean position
print("\n" + "="*70)
print("RESIDUALS FROM MEAN POSITION")
print("="*70)

# NZTM residuals
mean_e = mqzg['nztm_easting'].mean()
mean_n = mqzg['nztm_northing'].mean()
mean_u = mqzg['nzvd2016_elev'].mean()

mqzg['de_mm'] = (mqzg['nztm_easting'] - mean_e) * 1000
mqzg['dn_mm'] = (mqzg['nztm_northing'] - mean_n) * 1000
mqzg['du_mm'] = (mqzg['nzvd2016_elev'] - mean_u) * 1000

print("\nResidual Statistics (mm):")
print(f"East:  std = {mqzg['de_mm'].std():.3f} mm")
print(f"North: std = {mqzg['dn_mm'].std():.3f} mm")
print(f"Up:    std = {mqzg['du_mm'].std():.3f} mm")

# Calculate 3-D RMS
sigma_e = mqzg['de_mm'].std()
sigma_n = mqzg['dn_mm'].std()
sigma_u =mqzg['du_mm'].std()
rms_3d = np.sqrt(sigma_e**2 + sigma_n**2 + sigma_u**2)

print(f"\n3-D RMS: {rms_3d:.3f} mm")

# Component contributions
print(f"\nComponent contributions to 3-D RMS:")
print(f"East:  {(sigma_e**2 / rms_3d**2) * 100:.1f}%")
print(f"North: {(sigma_n**2 / rms_3d**2) * 100:.1f}%")
print(f"Up:    {(sigma_u**2 / rms_3d**2) * 100:.1f}%")

# Calculate daily 3-D magnitude
mqzg['rms_daily_mm'] = np.sqrt(mqzg['de_mm']**2 + mqzg['dn_mm']**2 + mqzg['du_mm']**2)

print("\n" + "="*70)
print("DAILY 3-D RMS STATISTICS")
print("="*70)
print(f"Mean:   {mqzg['rms_daily_mm'].mean():.3f} mm")
print(f"Median: {mqzg['rms_daily_mm'].median():.3f} mm")
print(f"Std:    {mqzg['rms_daily_mm'].std():.3f} mm")
print(f"Min:    {mqzg['rms_daily_mm'].min():.3f} mm")
print(f"Max:    {mqzg['rms_daily_mm'].max():.3f} mm")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Time series of residuals
ax = axes[0, 0]
ax.plot(mqzg['datetime'], mqzg['de_mm'], label='East', alpha=0.7)
ax.plot(mqzg['datetime'], mqzg['dn_mm'], label='North', alpha=0.7)
ax.plot(mqzg['datetime'], mqzg['du_mm'], label='Up', alpha=0.7)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Residuals (mm)')
ax.set_title('MQZG: Coordinate Residuals Over Time')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Daily 3-D RMS over time
ax = axes[0, 1]
ax.plot(mqzg['datetime'], mqzg['rms_daily_mm'], color='purple', alpha=0.7)
ax.axhline(mqzg['rms_daily_mm'].mean(), color='red', linestyle='--', 
           label=f'Mean: {mqzg["rms_daily_mm"].mean():.2f} mm')
ax.set_xlabel('Date')
ax.set_ylabel('Daily 3-D RMS (mm)')
ax.set_title('MGZG: Daily 3-D Positional Scatter')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: Horizontal scatter plot
ax = axes[1, 0]
ax.scatter(mqzg['de_mm'], mqzg['dn_mm'], alpha=0.5, s=20)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.axvline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('East Residual (mm)')
ax.set_ylabel('North Residual (mm)')
ax.set_title('MQZG: Horizontal Position Scatter')
ax.axis('equal')
ax.grid(alpha=0.3)

# Plot 4: Histogram of daily RMS
ax = axes[1, 1]
ax.hist(mqzg['rms_daily_mm'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(mqzg['rms_daily_mm'].mean(), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {mqzg["rms_daily_mm"].mean():.2f} mm')
ax.axvline(mqzg['rms_daily_mm'].median(), color='blue', linestyle='--', 
           linewidth=2, label=f'Median: {mqzg["rms_daily_mm"].median():.2f} mm')
ax.set_xlabel('Daily 3-D RMS (mm)')
ax.set_ylabel('Frequency')
ax.set_title('MQZG: Distribution of Daily 3-D RMS')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout
plt.savefig('exports/mqzg_coordinate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Analysis complete! Plot saved to exports/mqzg_coordinate_analysis.png")

# Check for outliers (>3 sigma)
outliers = mqzg[mqzg['rms_daily_mm'] > mqzg['rms_daily_mm'].mean() + 3*mqzg['rms_daily_mm'].std()]
if len(outliers) > 0:
    print(f"\n⚠️  Found {len(outliers)} potential outlier days (>3σ):")
    print(outliers[['date', 'rms_daily_mm']].to_string(index=False))

In [ ]:
# Load the WGTN data
wgtn = pd.read_csv('Exports/WGTN_GPS+Galileo+GLONASS_converted_coords.csv')

print("="*70)
print("WGTN DATA OVERVIEW")
print("="*70)
print(f"\nTotal observations: {len(wgtn)}")
print(f"Date range: {wgtn['date'].min()} to {wgtn['date'].max()}")
print(f"\nColumns: {wgtn.columns.tolist()}")

# Convert date to datetime for time series analysis
wgtn['datetime'] = pd.to_datetime(wgtn['date'])

print("\n" + "="*70)
print("COORDINATE STATISTICS")
print("="*70)

# ITRF2020 XYZ statistics
print("\nITRF2020 XYZ Coordinates (meters):")
print(f"X: mean = {wgtn['x'].mean():.4f}, std = {wgtn['x'].std():.6f}")
print(f"Y: mean = {wgtn['y'].mean():.4f}, std = {wgtn['y'].std():.6f}")
print(f"Z: mean = {wgtn['z'].mean():.4f}, std = {wgtn['z'].std():.6f}")

# NZTM coordinates statistics
print("\nNZTM Coordinates (meters):")
print(f"Easting:  mean = {wgtn['nztm_easting'].mean():.4f}, std = {wgtn['nztm_easting'].std():.6f}")
print(f"Northing: mean = {wgtn['nztm_northing'].mean():.4f}, std = {wgtn['nztm_northing'].std():.6f}")
print(f"Elevation: mean = {wgtn['nzvd2016_elev'].mean():.4f}, std = {wgtn['nzvd2016_elev'].std():.6f}")

# Calculate residuals from mean position
print("\n" + "="*70)
print("RESIDUALS FROM MEAN POSITION")
print("="*70)

# NZTM residuals
mean_e = wgtn['nztm_easting'].mean()
mean_n = wgtn['nztm_northing'].mean()
mean_u = wgtn['nzvd2016_elev'].mean()

wgtn['de_mm'] = (wgtn['nztm_easting'] - mean_e) * 1000
wgtn['dn_mm'] = (wgtn['nztm_northing'] - mean_n) * 1000
wgtn['du_mm'] = (wgtn['nzvd2016_elev'] - mean_u) * 1000

print("\nResidual Statistics (mm):")
print(f"East:  std = {wgtn['de_mm'].std():.3f} mm")
print(f"North: std = {wgtn['dn_mm'].std():.3f} mm")
print(f"Up:    std = {wgtn['du_mm'].std():.3f} mm")

# Calculate 3-D RMS
sigma_e = wgtn['de_mm'].std()
sigma_n = wgtn['dn_mm'].std()
sigma_u = wgtn['du_mm'].std()
rms_3d = np.sqrt(sigma_e**2 + sigma_n**2 + sigma_u**2)

print(f"\n3-D RMS: {rms_3d:.3f} mm")

# Component contributions
print(f"\nComponent contributions to 3-D RMS:")
print(f"East:  {(sigma_e**2 / rms_3d**2) * 100:.1f}%")
print(f"North: {(sigma_n**2 / rms_3d**2) * 100:.1f}%")
print(f"Up:    {(sigma_u**2 / rms_3d**2) * 100:.1f}%")

# Calculate daily 3-D magnitude
wgtn['rms_daily_mm'] = np.sqrt(wgtn['de_mm']**2 + wgtn['dn_mm']**2 + wgtn['du_mm']**2)

print("\n" + "="*70)
print("DAILY 3-D RMS STATISTICS")
print("="*70)
print(f"Mean:   {wgtn['rms_daily_mm'].mean():.3f} mm")
print(f"Median: {wgtn['rms_daily_mm'].median():.3f} mm")
print(f"Std:    {wgtn['rms_daily_mm'].std():.3f} mm")
print(f"Min:    {wgtn['rms_daily_mm'].min():.3f} mm")
print(f"Max:    {wgtn['rms_daily_mm'].max():.3f} mm")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Time series of residuals
ax = axes[0, 0]
ax.plot(wgtn['datetime'], wgtn['de_mm'], label='East', alpha=0.7)
ax.plot(wgtn['datetime'], wgtn['dn_mm'], label='North', alpha=0.7)
ax.plot(wgtn['datetime'], wgtn['du_mm'], label='Up', alpha=0.7)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Residuals (mm)')
ax.set_title('WGTN: Coordinate Residuals Over Time')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Daily 3-D RMS over time
ax = axes[0, 1]
ax.plot(wgtn['datetime'], wgtn['rms_daily_mm'], color='purple', alpha=0.7)
ax.axhline(wgtn['rms_daily_mm'].mean(), color='red', linestyle='--',
           label=f'Mean: {wgtn["rms_daily_mm"].mean():.2f} mm')
ax.set_xlabel('Date')
ax.set_ylabel('Daily 3-D RMS (mm)')
ax.set_title('WGTN: Daily 3-D Positional Scatter')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: Horizontal scatter plot
ax = axes[1, 0]
ax.scatter(wgtn['de_mm'], wgtn['dn_mm'], alpha=0.5, s=20)
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.axvline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('East Residual (mm)')
ax.set_ylabel('North Residual (mm)')
ax.set_title('WGTN: Horizontal Position Scatter')
ax.axis('equal')
ax.grid(alpha=0.3)

# Plot 4: Histogram of daily RMS
ax = axes[1, 1]
ax.hist(wgtn['rms_daily_mm'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(wgtn['rms_daily_mm'].mean(), color='red', linestyle='--',
           linewidth=2, label=f'Mean: {wgtn["rms_daily_mm"].mean():.2f} mm')
ax.axvline(wgtn['rms_daily_mm'].median(), color='blue', linestyle='--',
           linewidth=2, label=f'Median: {wgtn["rms_daily_mm"].median():.2f} mm')
ax.set_xlabel('Daily 3-D RMS (mm)')
ax.set_ylabel('Frequency')
ax.set_title('WGTN: Distribution of Daily 3-D RMS')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exports/wgtn_coordinate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Analysis complete! Plot saved to exports/wgtn_coordinate_analysis.png")

# Check for outliers (>3 sigma)
outliers = wgtn[wgtn['rms_daily_mm'] > wgtn['rms_daily_mm'].mean() + 3*wgtn['rms_daily_mm'].std()]
if len(outliers) > 0:
    print(f"\n⚠️  Found {len(outliers)} potential outlier days (>3σ):")
    print(outliers[['date', 'rms_daily_mm']].to_string(index=False))
